# Past / Current Service IO（正式命名版）

本版已將沈同學的 Past / Current Service 重新整理成 **Statistics Service 一致命名格式**。

## 這版的核心原則

本版不再使用「對應表」作為主要整合方式，而是直接把 Past / Current Service 的 input / output 欄位改成 正式命名。

也就是直接使用：

- `stations`
- `line_id`
- `station_name_zh`
- `station_name_en`
- `state`
- `metrics`
- `process_parameters`

不再使用：

- `machines`
- `machine_id`
- `process_name`
- `Qc`
- `Ut`
- `para`
- `cycle_time`

## 整體流程

```text
Past Service
 ↓
Current Service
 ↓
Future Service
 ↓
Statistics Service
 ↓
UI(E) / UI(M)
 ↓
Ontology / Runtime TTL
```


## 0.1 本版小補強內容

本版以原本「正式命名版」為主，不加入額外噴漆三大類欄位。 
本次只做小補強，目的是讓 Past / Current Service 更好銜接 Statistics Service 與後續 ontology。


1. Past / Current output 新增 `generated_at`，表示 service 產生此份 output 的時間。
2. Past / Current window 新增 `window_id`，方便後續 ontology / TTL 建立 snapshot individual。
3. Past window 補 `window_start`、`window_end`，保留歷史視窗的時間範圍。
4. Current window 補 `display_label`，並保留 `window_size = defined_by_system`，不先寫死。
5. 補上 state vocabulary 說明：`Running`、`Standby`、`Stop`、`Maintenance`、`Alarm`。
6. 補上欄位責任說明：哪些欄位由 Past / Current 直接提供，哪些欄位由 Statistics / Rule / Future Service 補齊。
7. 保留待確認事項：Past sample method、Current window size、Current sample method、flow rate 單位。


## 1. 站別命名統一

三站統一使用以下 正式命名：

| line_id | station_name_zh | station_name_en |
|---|---|---|
| line_1 | 底漆站 | Primer Station |
| line_2 | 面漆站 | Topcoat Station |
| line_3 | 金漆站 | Gold Paint Station |

> 若原始資料曾使用「色漆」，本版一律統一為「面漆站 / Topcoat Station」。


## 2. Past Service Input

Past Service 負責查詢一段歷史範圍內的站別資料。 
本版 input 已改成 風格命名，使用 `stations[]`，不再使用 `machines[]`。


In [ ]:
past_service_input = {
 "service_name": "past_service",
 "schema_version": "service-contract-compatible",
 "range_variable": {
 "type": "history_range",
 "description": "Past Service 查詢的歷史範圍"
 },
 "window": {
 "mode": "time_or_batch",
 "window_type": "2hour_or_10batch",
 "window_size": "defined_by_system"
 },
 "sample_method": "defined_by_service",
 "target": [
 "state",
 "metrics",
 "process_parameters"
 ],
 "stations": [
 {
 "line_id": "line_1",
 "station_name_zh": "底漆站",
 "station_name_en": "Primer Station"
 },
 {
 "line_id": "line_2",
 "station_name_zh": "面漆站",
 "station_name_en": "Topcoat Station"
 },
 {
 "line_id": "line_3",
 "station_name_zh": "金漆站",
 "station_name_en": "Gold Paint Station"
 }
 ]
}

past_service_input


## 3. Past Service Output

Past Service output 已改成 一致命名。 
重點是把原本的 `Qc`、`Ut`、`para`、`cycle_time` 全部整理進：

- `metrics`
- `process_parameters`

### 欄位命名

| 原概念 | 統一欄位 |
|---|---|
| 品質 Qc | `metrics.quality_score_pct` |
| 利用率 Ut | `process_parameters.utilization_pct` |
| 壓力 | `metrics.pressure_bar` |
| 流量 | `metrics.flow_rate_ml_min` |
| 溫度 | `process_parameters.temperature_c` |
| Cycle time | `process_parameters.cycle_time_sec` |

> `flow_rate_ml_min` 這個欄位名稱先與 / UI 對齊；實際資料進入時，上游需確認或換算成 ml/min 後再填入。


In [ ]:
past_service_output = {
 "service_name": "past_service",
 "schema_version": "service-contract-compatible",
 "generated_at": "<datetime-string>",
 "window": {
 "window_id": "<string>",
 "mode": "time_or_batch",
 "window_type": "2hour_or_10batch",
 "window_start": "<datetime-string|null>",
 "window_end": "<datetime-string|null>",
 "display_label": "past window"
 },
 "stations": [
 {
 "line_id": "line_1",
 "station_name_zh": "底漆站",
 "station_name_en": "Primer Station",
 "state": "Running",
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 }
 },
 {
 "line_id": "line_2",
 "station_name_zh": "面漆站",
 "station_name_en": "Topcoat Station",
 "state": "Running",
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 }
 },
 {
 "line_id": "line_3",
 "station_name_zh": "金漆站",
 "station_name_en": "Gold Paint Station",
 "state": "Maintenance",
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 }
 }
 ],
 "notes": [
 "Past Service 主要提供歷史視窗內的站別狀態、metrics 與 process_parameters。",
 "availability_pct、clog_rate_pct、maintainability_pct、risk_text 若不是 Past Service 直接產生，可先保留 null，由後續 Statistics / Rule / Future Service 補齊。"
 ]
}

past_service_output


## 4. Current Service Input

Current Service 負責取得目前或近即時的站別資料。 
本版同樣使用 一致命名，不再使用 `machine_id` 或 `machines`。


In [ ]:
current_service_input = {
 "service_name": "current_service",
 "schema_version": "service-contract-compatible",
 "window": {
 "mode": "current",
 "window_type": "current_window",
 "window_size": "defined_by_system",
 "display_label": "current"
 },
 "sample_method": "defined_by_service",
 "target": [
 "state",
 "metrics",
 "process_parameters"
 ],
 "stations": [
 {
 "line_id": "line_1",
 "station_name_zh": "底漆站",
 "station_name_en": "Primer Station"
 },
 {
 "line_id": "line_2",
 "station_name_zh": "面漆站",
 "station_name_en": "Topcoat Station"
 },
 {
 "line_id": "line_3",
 "station_name_zh": "金漆站",
 "station_name_en": "Gold Paint Station"
 }
 ]
}

current_service_input


## 5. Current Service Output

Current Service output 也已改成 一致命名。 
此處的資料可直接作為 Statistics Service 的上游輸入之一。

> 本版不再另外做 mapping table，因為欄位名稱本身已經對齊 。


In [ ]:
current_service_output = {
 "service_name": "current_service",
 "schema_version": "service-contract-compatible",
 "generated_at": "<datetime-string>",
 "window": {
 "window_id": "<string>",
 "mode": "current",
 "window_type": "current_window",
 "window_size": "defined_by_system",
 "display_label": "current"
 },
 "stations": [
 {
 "line_id": "line_1",
 "station_name_zh": "底漆站",
 "station_name_en": "Primer Station",
 "state": "Running",
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 }
 },
 {
 "line_id": "line_2",
 "station_name_zh": "面漆站",
 "station_name_en": "Topcoat Station",
 "state": "Running",
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 }
 },
 {
 "line_id": "line_3",
 "station_name_zh": "金漆站",
 "station_name_en": "Gold Paint Station",
 "state": "Maintenance",
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 }
 }
 ],
 "notes": [
 "Current Service 提供目前站別狀態、metrics 與 process_parameters。",
 "若某些風險或維護指標不是 Current Service 直接產生，可先保留 null，交由後續 Statistics / Rule / Future Service 補齊。"
 ]
}

current_service_output


## 6. Past + Current 給 Statistics Service 的整合輸出

因為 Past / Current 現在已經直接使用 一致命名，所以這裡不需要額外 mapping。 
Statistics Service 可以直接讀取：

```text
past_current_integrated_output.past.stations[]
past_current_integrated_output.current.stations[]
```


In [ ]:
past_current_integrated_output = {
 "schema_version": "service-contract-compatible",
 "description": "Past / Current Service 使用 一致命名後，提供給 Statistics Service 的整合輸入",
 "past": past_service_output,
 "current": current_service_output,
 "next_service": {
 "statistics_service": {
 "expected_input": [
 "generated_at",
 "window.window_id",
 "window.window_type",
 "stations[].line_id",
 "stations[].station_name_zh",
 "stations[].station_name_en",
 "stations[].state",
 "stations[].metrics",
 "stations[].process_parameters"
 ],
 "to_be_completed_by_statistics_or_rule_service": [
 "summary",
 "component_overview",
 "trend_series",
 "availability_pct",
 "clog_rate_pct",
 "maintainability_pct",
 "risk_text"
 ]
 },
 "future_service": {
 "expected_input": [
 "past.stations[]",
 "current.stations[]",
 "target",
 "range",
 "confidence_level"
 ],
 "note": "Future Service 尚未定案，此處只保留可能需要的輸入方向。"
 }
 }
}

past_current_integrated_output


## 欄位標準與責任說明

### 1. State vocabulary

目前 `state` 建議統一使用下列狀態名稱：

```text
Running
Standby
Stop
Maintenance
Alarm
```

這些狀態名稱後續可在 ontology 中建成 `StationState` 的 individual。 
JSON / schema 目前不強制鎖死 enum，先保持彈性；但進入 ontology 前，建議統一到上述狀態 vocabulary。

### 2. Past / Current 比較可能直接提供的欄位

```text
state
pressure_bar
flow_rate_ml_min
quality_score_pct
temperature_c
utilization_pct
cycle_time_sec
```

這些比較接近 Past / Current 上游資料可以提供的站別狀態、量測指標或製程參數。

### 3. 較可能由 Statistics / Rule / Future Service 補齊的欄位

```text
availability_pct
clog_rate_pct
maintainability_pct
risk_text
component_overview
summary
trend_series
```

因此 Past / Current output 中可以保留這些欄位位置，但不應強制 Past / Current Service 一定要產生完整值。 
如果目前尚無資料，可先使用 `<number|null>` 或 `<string|null>`。


## 7. 本版最終整理

### Past Service

```text
Input:
stations[]
window
sample_method
target

Output:
stations[]
state
metrics
process_parameters
```

### Current Service

```text
Input:
stations[]
window
sample_method
target

Output:
stations[]
state
metrics
process_parameters
```

### 與 對齊後的好處

1. 不需要再另外用對應表把 `machine_id` 改成 `line_id`。
2. 不需要再把 `Qc` 改成 `quality_score_pct`，因為一開始就用 正式命名。
3. 不需要再把 `Ut` 改成 `utilization_pct`，因為一開始就用 正式命名。
4. Past / Current 可以直接被 Statistics Service 讀取。
5. UI(E) 和 ontology 後續接資料時，欄位名稱不會再打架。

### 最終流程

```text
Past Service
 ↓
Current Service
 ↓
Statistics Service
 ↓
UI(E) / UI(M)
 ↓
Ontology / Runtime TTL
```


## 本次小補強後的銜接重點

```text
Past / Current Service
 ↓
提供 station snapshots、window_id、generated_at
 ↓
Statistics Service
 ↓
產生 summary、component_overview、trend_series、risk-related metrics
 ↓
UI(E) / Ontology / Runtime TTL
```

本版仍以原版未補強資料結構為主，不加入額外噴漆三大類欄位。
